# Figure 3 — resampling the upstream signal

Two controls on random-label pretraining, measured the same way: downstream $t_0$ (epochs to
memorize CIFAR-10 random labels, `label_seed=42`) after warm-starting from each pretraining epoch.
(a) **Label resampling** (natural-image upstream) — fixed vs redrawn-every-epoch labels.
(b) **Data resampling** (random-pixel upstream, label held fixed) — fixed vs redrawn-every-epoch
pixels. If pretraining only helped by *exposing* inputs, resampling would not matter; it does.

- **Consumes:** `downstream_randlabel_from_randpixel_resample/` (raw, panel b resampled — refit here
  with `common.fit_t0`); `_cache/label_resampling_comparison_t0.csv` (panel a) and
  `_cache/fig1c_randpixel_finegrained_t0.csv` (panel b fixed pixels), written by the sibling
  notebooks that own those experiments.
- **Produces:** `outputs/fig3_resampling.png`


In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_root = Path.cwd()
while not (_root / "common").is_dir() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

from common.config import FIGURE_DATA_DIR, OUTPUT_DIR
from common.fit_t0 import fit_t0

BASELINE = 41.12  # shared from-scratch -> randlabel42 t0 anchor (fixed-label epoch-0 run)
RES_DIR = FIGURE_DATA_DIR / "downstream_randlabel_from_randpixel_resample"
CACHE = FIGURE_DATA_DIR / "_cache"


def t0_of(metrics_path):
    m = json.loads(Path(metrics_path).read_text(encoding="utf-8"))
    return fit_t0([r["epoch"] for r in m["epochs"]], [r["train_acc"] for r in m["epochs"]]).t0

In [ ]:
# panel (b) resampled pixels: refit the 19 raw runs with the shared fit_t0
res = pd.DataFrame(
    [dict(pretraining_epoch=k, t0=t0_of(RES_DIR / f"epoch{k:04d}" / "metrics.json")) for k in range(19)]
).sort_values("pretraining_epoch").reset_index(drop=True)

# panel (a) + panel (b) fixed pixels: from the caches their own notebooks wrote
lab = pd.read_csv(CACHE / "label_resampling_comparison_t0.csv")
keep = pd.read_csv(CACHE / "fig1c_randpixel_finegrained_t0.csv").sort_values("pretraining_epoch")
print("resampled-pixel trough: t0=%.2f at epoch %d"
      % (res.t0.min(), int(res.pretraining_epoch[res.t0.idxmin()])))
print("fixed-pixel   trough: t0=%.2f at epoch %d"
      % (keep.t0.min(), int(keep.pretraining_epoch[keep.t0.idxmin()])))

In [ ]:
FS_TITLE = 13
plt.rcParams.update({
    "figure.dpi": 130, "font.family": "DejaVu Sans",
    "axes.labelsize": 12, "xtick.labelsize": 11, "ytick.labelsize": 11,
    "legend.fontsize": 11, "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.9,
})
C_FIX, C_RES, C_BASE = plt.cm.viridis(0.15), plt.cm.viridis(0.60), "#c44e52"
YLIM = (0, 58)

labF = lab[lab.curve == "Fixed labels"].sort_values("pretraining_epoch")
labR = lab[lab.curve == "Resampled labels"].sort_values("pretraining_epoch")
xL = labF.pretraining_epoch.to_numpy(float)
yLf, yLr = labF.t0_plotted.to_numpy(float), labR.t0_plotted.to_numpy(float)

xR = keep.pretraining_epoch.to_numpy(float)
anchor = lambda df: np.array([BASELINE if int(e) == 0 else t for e, t in zip(df.pretraining_epoch, df.t0)], float)
yRk, yRr = anchor(keep), anchor(res)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.5))

axL.plot([0, xL.max() + 1.5], [BASELINE, BASELINE], ls="--", lw=1.2, color=C_BASE, zorder=1, label="From-scratch baseline")
axL.plot([0, xL.max()], [BASELINE, BASELINE - xL.max()], ls="--", lw=1.3, color="0.5", zorder=1, label="slope = \u22121 (ref.)")
axL.plot(xL, yLf, "-o", ms=6, lw=1.8, color=C_FIX, mec="white", mew=0.6, zorder=3, label="Fixed labels")
axL.plot(xL, yLr, "-s", ms=6, lw=1.8, color=C_RES, mec="white", mew=0.6, zorder=4, label="Resampled labels")
axL.set_xlim(-1.2, xL.max() + 1.5); axL.set_ylim(*YLIM); axL.set_xticks([0, 10, 20, 30, 40, 47])
axL.set_xlabel("Pre-training epoch"); axL.set_ylabel(r"$t_0$ (downstream epochs)")
axL.set_title("(a) Label resampling", fontsize=FS_TITLE, pad=8)
axL.legend(frameon=False, loc="lower left")

axR.plot([0, 18.6], [BASELINE, BASELINE], ls="--", lw=1.2, color=C_BASE, zorder=1, label="From-scratch baseline")
axR.plot([0, xR.max()], [BASELINE, BASELINE - xR.max()], ls="--", lw=1.3, color="0.5", zorder=1, label="slope = \u22121 (ref.)")
axR.plot(xR, yRk, "-o", ms=5, lw=1.8, color=C_FIX, mec="white", mew=0.6, zorder=3, label="Fixed pixels")
axR.plot(xR, yRr, "-s", ms=5, lw=1.8, color=C_RES, mec="white", mew=0.6, zorder=4, label="Resampled pixels")
axR.set_xlim(-0.6, 18.6); axR.set_ylim(*YLIM); axR.set_xticks([0, 3, 6, 9, 12, 15, 18])
axR.set_xlabel("Pre-training epoch")
axR.set_title("(b) Data resampling (random pixels)", fontsize=FS_TITLE, pad=8)
axR.legend(frameon=False, loc="upper right")

fig.tight_layout()
OUT_PNG = OUTPUT_DIR / "fig3_resampling.png"
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_PNG, dpi=300, bbox_inches="tight")
print("saved", OUT_PNG)